# Training behavior and scaling choices

Small models make training decisions visible. We compare width and context length under a fixed parameter budget style, then inspect underfitting and overfitting instead of choosing a single favorable run.

In [1]:
import math
import random
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch import nn
from torch.nn import functional as F
SEED = 7
random.seed(SEED)
torch.manual_seed(SEED)
sns.set_theme(style='whitegrid', context='notebook')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

Matplotlib is building the font cache; this may take a moment.


device: cpu


In [2]:
def make_synthetic_data(length=5000):
    text = ('the quick brown fox jumps over the lazy dog. ' * 200).lower()
    chars = sorted(set(text))
    enc = {c: i for i, c in enumerate(chars)}
    ids = torch.tensor([enc[c] for c in text], dtype=torch.long)
    cut = int(0.8 * len(ids))
    return (ids[:cut], ids[cut:], len(chars))
train_stream, val_stream, vocab = make_synthetic_data()
print('vocabulary:', vocab)

vocabulary: 28


The controlled comparison below changes one setting at a time. The curves answer practical questions: does a longer context help this corpus, does width reduce training loss, and when does validation stop improving?

In [3]:
class SmallContextModel(nn.Module):

    def __init__(self, vocabulary_size, context_length, embedding_size=16, hidden_size=64):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, embedding_size)
        self.network = nn.Sequential(nn.Linear(context_length * embedding_size, hidden_size), nn.Tanh(), nn.Linear(hidden_size, vocabulary_size))

    def forward(self, x, y=None):
        logits = self.network(self.embedding(x).reshape(x.size(0), -1))
        loss = None if y is None else F.cross_entropy(logits, y)
        return (logits, loss)

def windows(stream, context_length):
    x = torch.stack([stream[i:i + context_length] for i in range(len(stream) - context_length)])
    y = stream[context_length:]
    return (x, y)

def run_experiment(context_length, hidden_size, steps=250):
    x, y = windows(train_stream, context_length)
    vx, vy = windows(val_stream, context_length)
    model = SmallContextModel(vocab, context_length, hidden_size=hidden_size).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.003)
    history = []
    generator = torch.Generator().manual_seed(SEED + context_length + hidden_size)
    for step in range(steps):
        idx = torch.randint(len(x), (128,), generator=generator)
        xb, yb = (x[idx].to(DEVICE), y[idx].to(DEVICE))
        _, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        if step % 50 == 0:
            with torch.no_grad():
                history.append(float(model(vx[:512].to(DEVICE), vy[:512].to(DEVICE))[1]))
    return (history[-1], history)
configurations = {'context 2': (2, 64), 'context 8': (8, 64), 'wider hidden': (8, 128)}
results = {label: run_experiment(*config) for label, config in configurations.items()}
for label, (final_loss, _) in results.items():
    print(label, 'final validation loss', round(final_loss, 3))

context 2 final validation loss 0.051
context 8 final validation loss 0.005
wider hidden final validation loss 0.002


plt.figure(figsize=(8,4))
for label,(final_loss,history) in results.items(): plt.plot(range(0,len(history)*50,50),history,marker="o",label=label)
plt.xlabel("training step"); plt.ylabel("validation loss"); plt.title("Measured one-factor comparisons"); plt.legend(); plt.show()

## What a fair comparison requires

Changing context length, width, or layer count also changes parameter count and compute. The experiment therefore records the configuration, uses the same ordered split, optimizer family, seed policy, and training-step budget, and compares validation loss rather than only training loss. Otherwise a larger or longer run can appear better simply because it received more opportunities to fit.

## Reading the curves

A falling training loss with a flat or rising validation loss is evidence of overfitting. Two curves that are both high indicate underfitting or an optimization problem. A gap between configurations is useful only together with the configuration table: the plot shows what happened, while the table explains what was changed.

## Scope of the result

These experiments use a tiny synthetic corpus and a small feed-forward family. They demonstrate an experimental method, not a universal scaling law. Larger claims require multiple seeds, a wider range of budgets, and a separation between architecture effects and optimization effects.

These are measured on the same synthetic corpus with the same optimizer and step budget. The comparison is still narrow: one corpus and one small model family cannot establish a general scaling law. It does show the reporting discipline needed before making a larger claim.